In [1]:
!git clone https://github.com/Ricy-Labs/mmm-kahf-indonesia.git
%cd mmm-kahf-indonesia
!git checkout feat/ricy-pipeline
!pip install -q pymc-marketing

from src.data_loader import load_data
from src.config import CHANNEL_COLS, CONTROL_COLS, DATE_COL, TARGET_COL
from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation

df = load_data()
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].rename("y")

mmm = MMM(
    date_column=DATE_COL,
    channel_columns=CHANNEL_COLS,
    control_columns=CONTROL_COLS,
    adstock=GeometricAdstock(l_max=28),
    saturation=LogisticSaturation(),
    yearly_seasonality=2,
)
mmm.fit(X, y, chains=2, draws=500, tune=500, target_accept=0.9, random_seed=42)
mmm.save("mmm_kahf.nc")

Cloning into 'mmm-kahf-indonesia'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 48 (delta 11), reused 19 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 25.92 KiB | 1.13 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/mmm-kahf-indonesia
branch 'feat/ricy-pipeline' set up to track 'origin/feat/ricy-pipeline'.
Switched to a new branch 'feat/ricy-pipeline'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.1/771.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.4/209.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.7/527.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.3/608.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

Output()

/usr/local/lib/python3.13/dist-packages/pymc/sampling/deterministic.py:143: FutureWarning: `merge_dataset` is deprecated and will be removed in a future release. Use `extend_dataset=True` to add the deterministics to the original dataset, passing a `.copy()` of it if it should not be mutated.
  warnings.warn(


Output()

In [5]:
import arviz as az
import pandas as pd

print("Divergences:", int(mmm.idata.sample_stats["diverging"].sum()))
print(az.summary(mmm.idata, var_names=["saturation_beta", "saturation_lam", "adstock_alpha"]))

post = mmm.idata.posterior
if "channel_contribution_original_scale" in post:
    contrib = post["channel_contribution_original_scale"]
else:
    contrib = post["channel_contribution"] * mmm.idata.constant_data["target_scale"]

est = contrib.mean(("chain", "draw")).sum("date")
est = (est / df["sales"].sum()).to_series().rename("estimated_share")

truth = pd.read_csv("data/processed/true_contributions.csv").set_index("channel")
result = truth.join(est)
result["selisih"] = result["estimated_share"] - result["true_contribution_share"]
print(result.round(3))

Divergences: 0
                                        mean     sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
saturation_beta[influencer_S]           0.23    0.3    0.055     0.62      265      269  1.01     0.016    0.063
saturation_beta[instagram_ads_S]         0.1   0.18   0.0065     0.32      270      262  1.01    0.0099    0.039
saturation_beta[marketplace_ads_S]      0.21   0.09     0.15     0.32      300      204  1.01    0.0054    0.017
saturation_beta[tiktok_ads_S]          0.075   0.08    0.014     0.16      347      432  1.00    0.0039    0.011
saturation_beta[offline_activation_S]   0.31   0.38    0.055     0.86      569      576  1.00     0.015    0.039
saturation_lam[influencer_S]             3.3    2.5     0.55      8.5      381      411  1.01      0.13     0.12
saturation_lam[instagram_ads_S]          2.7    2.2     0.37      6.8      652      438  1.00     0.072      0.1
saturation_lam[marketplace_ads_S]        4.4    1.6      1.8      6.8      321   